# 🛡️ CNN Phân Loại Mã Độc — PTQ INT8 **FIXED** (Chi-You Style)

## 📌 Khác biệt so với notebook cũ:
- ✅ **Kiến trúc khôngBN**: Conv2D → ReLU → MaxPool (đơn giản, không fold BN phức tạp)
- ✅ **Calibration Entropy-based**: Dùng entropy / percentile thay vì max → tránh outliers
- ✅ **Shift bits adaptive**: Thử nhiều shift_bits → chọn tối ưu
- ✅ **Debug layer-by-layer**: So sánh float32 vs INT8 từng layer
- ✅ **Verify kỹ càng**: Simulate INT8 hardware giống 100%

---

## ⚠️ Nguyên nhân accuracy drop 71.5% trong notebook cũ:
1. **Shift bits sai** → INT32 accumulator >> quá nhiều → toàn 0
2. **Calibration dùng max** → outliers làm range quá rộng
3. **Quantization quá aggressive** → mất quá nhiều precision
4. **Không debug từng layer** → khó biết lỗi ở đâu

---

## 📦 1. Cài Đặt & Mount

In [1]:
!pip install -q openpyxl scikit-learn

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## ⚙️ 2. Import + Config


In [3]:
import os, glob, random, time, json, warnings
import numpy as np
import tensorflow as tf
import keras
from keras import layers, Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from keras.optimizers import Adam
from PIL import Image
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# ── Config ──────────────────────────────────────────
DATASET_PATH = '/content/drive/MyDrive/CNX/malimg'
OUTPUT_DIR   = '/content/drive/MyDrive/CNX'

BATCH_SIZE   = 32
EPOCHS_F32   = 50
TEST_SPLIT   = 0.3
CALIB_SIZE   = 200
Q_MAX        = 127

print('✅ Libraries loaded!')

✅ Libraries loaded!


## 📚 3. Load Dataset


In [4]:
# [Giữ code load dataset từ notebook cũ - tương tự]
# Load từ cache hoặc đọc từ folder

print(f'Loading dataset from {DATASET_PATH}...')
benign_paths, malware_paths = [], []

for fam_name in os.listdir(DATASET_PATH):
    fam_dir = os.path.join(DATASET_PATH, fam_name)
    if not os.path.isdir(fam_dir): continue
    imgs = glob.glob(os.path.join(fam_dir, '*.png'))
    if fam_name.lower() == 'benign': benign_paths.extend(imgs)
    else: malware_paths.extend(imgs)

num_benign = len(benign_paths)
target_malware = num_benign * 2
if len(malware_paths) > target_malware:
    malware_paths = random.sample(malware_paths, target_malware)

print(f'Benign: {len(benign_paths)}, Malware: {len(malware_paths)}')

X, y = [], []
for p in benign_paths:
    try:
        with Image.open(p) as im:
            X.append(np.array(im.convert('L').resize((32, 8), Image.Resampling.LANCZOS)) / 255.0)
            y.append([1, 0])  # benign
    except: pass

for p in malware_paths:
    try:
        with Image.open(p) as im:
            X.append(np.array(im.convert('L').resize((32, 8), Image.Resampling.LANCZOS)) / 255.0)
            y.append([0, 1])  # malware
    except: pass

X = np.array(X, dtype=np.float32).reshape(-1, 8, 32, 1)
y = np.array(y, dtype=np.float32)

# Split train/calib/test
n_total = len(X)
n_test = int(n_total * TEST_SPLIT)
n_train_calib = n_total - n_test

idx = np.arange(n_total)
np.random.shuffle(idx)

X_train_calib = X[idx[:n_train_calib]]
y_train_calib = y[idx[:n_train_calib]]
X_test = X[idx[n_train_calib:]]
y_test = y[idx[n_train_calib:]]

# Tách calibration
X_train = X_train_calib[:-CALIB_SIZE]
y_train = y_train_calib[:-CALIB_SIZE]
X_calib = X_train_calib[-CALIB_SIZE:]
y_calib = y_train_calib[-CALIB_SIZE:]

print(f'Train: {X_train.shape}, Calib: {X_calib.shape}, Test: {X_test.shape}')

Loading dataset from /content/drive/MyDrive/CNX/malimg...
Benign: 982, Malware: 1964
Train: (1863, 8, 32, 1), Calib: (200, 8, 32, 1), Test: (883, 8, 32, 1)


## 🏗️ 4. Build Model Float32 (KHÔNG BatchNorm)


In [9]:
def build_model_fixed(input_shape=(8, 32, 1), num_classes=2):
    """
    Model FIXED — Không có BatchNorm (đơn giản hơn, dễ quantize).

    Kiến trúc:
      Input(8, 32, 1)
        ↓
      Conv2D(16, 3×3) → ReLU → MaxPool(2×2)
        ↓ (6, 15, 16) → tại sao 15? vì (32-3)/1+1 = 30, nhưng maxpool chia 2 → 15
      Conv2D(32, 3×3) → ReLU → Flatten
        ↓ (4, 13, 32) → flatten → 1664
      Dense(48) → ReLU
        ↓
      Dense(2) → output logits
    """
    # Explicitly define the input tensor for a functional model
    inputs = keras.Input(shape=input_shape)

    # Build the layers using the functional API style
    x = Conv2D(16, (3, 3), strides=(1,1), padding='valid',
               activation='relu', name='Conv2D_1')(inputs) # Use the defined inputs
    x = MaxPooling2D((2, 2), name='MaxPool_1')(x)

    x = Conv2D(32, (3, 3), strides=(1,1), padding='valid',
               activation='relu', name='Conv2D_2')(x)

    x = Flatten(name='Flatten')(x)
    x = Dense(48, activation='relu', name='Dense_1')(x)
    outputs = Dense(2, activation=None, name='Dense_2')(x)  # logits, no softmax

    # Create a functional model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

model_f32 = build_model_fixed()
model_f32.compile(
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)
model_f32.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 8, 32, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_1 (Conv2D)               │ (None, 6, 30, 16)      │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_1 (MaxPooling2D)        │ (None, 3, 15, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2D_2 (Conv2D)               │ (None, 1, 13, 32)      │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 416)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_1 (Dense)                 │ (None, 48)             │        20,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_2 (Dense)                 │ (None, 2)              │            98 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,914 (97.32 KB)

 Trainable params: 24,914 (97.32 KB)

 Non-trainable params: 0 (0.00 B)

## 🏋️ 5. Train Float32


In [10]:
print('Training Float32 model...')
history = model_f32.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_F32,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ],
    verbose=1
)

# Evaluate
loss_f32, acc_f32 = model_f32.evaluate(X_test, y_test, verbose=0)
print(f'\n✅ Float32 Test Accuracy: {acc_f32*100:.2f}%')

Training Float32 model...
Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6599 - loss: 0.6422 - val_accuracy: 0.6631 - val_loss: 0.6100 - learning_rate: 0.0010
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7064 - loss: 0.5337 - val_accuracy: 0.8503 - val_loss: 0.4060 - learning_rate: 0.0010
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8890 - loss: 0.3198 - val_accuracy: 0.9679 - val_loss: 0.1702 - learning_rate: 0.0010
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9487 - loss: 0.1767 - val_accuracy: 0.9733 - val_loss: 0.0965 - learning_rate: 0.0010
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9570 - loss: 0.1512 - val_accuracy: 0.9733 - val_loss: 0.0788 - learning_rate: 0.0010
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9618 - loss: 0.1319 - val_accuracy: 0.9786 - val_loss: 0.0732 - learning_rate: 0.0010
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9612 - 

## 📊 6. CALIBRATION (Entropy-Based)

In [11]:
print('='*70)
print('CALIBRATION — Entropy-Based Quantization')
print('='*70)

# Extract activations từng layer
layer_names_relu = ['Conv2D_1', 'Conv2D_2', 'Dense_1']

# Explicitly build the model if not already built
if not model_f32.built:
    # Assuming the input shape is (None, 8, 32, 1) as defined in build_model_fixed
    model_f32.build(input_shape=(None, 8, 32, 1))

activation_extractor = keras.Model(
    inputs=model_f32.input,
    outputs=[model_f32.get_layer(n).output for n in layer_names_relu]
)

print(f'\n📊 Extract activations từ {CALIB_SIZE} calibration samples...')
acts = activation_extractor.predict(X_calib, batch_size=32, verbose=0)

# Analyze activations
calib_stats = {}
print(f'\n{"Layer":<12} {"Min":>8} {"Max":>8} {"Mean":>8} {"Std":>8} {"P99":>8}')
print('-'*60)

for i, name in enumerate(layer_names_relu):
    act = acts[i].flatten()
    calib_stats[name] = {
        'min': float(act.min()),
        'max': float(act.max()),
        'mean': float(act.mean()),
        'std': float(act.std()),
        'p99': float(np.percentile(act, 99))
    }
    print(f'{name:<12} {act.min():>8.3f} {act.max():>8.3f} {act.mean():>8.3f} {act.std():>8.3f} {np.percentile(act, 99):>8.3f}')

print('\n✅ Calibration stats ready')

CALIBRATION — Entropy-Based Quantization

📊 Extract activations từ 200 calibration samples...

Layer             Min      Max     Mean      Std      P99
------------------------------------------------------------
Conv2D_1        0.000    0.317    0.019    0.039    0.168
Conv2D_2        0.000    0.973    0.067    0.109    0.448
Dense_1         0.000    8.719    0.605    1.171    6.150

✅ Calibration stats ready


## 🔧 7. Quantize Weights (Simple Version)


In [12]:
print('='*70)
print('QUANTIZE WEIGHTS → INT8')
print('='*70)

def quantize_weights_symmetric(w_float32, q_max=127):
    """
    Symmetric INT8 quantization:
      scale = max(|w|) / q_max
      w_int8 = round(w / scale)
    """
    w_max = np.max(np.abs(w_float32))
    scale = w_max / q_max
    w_int8 = np.round(w_float32 / scale).astype(np.int8)
    return w_int8, scale

# Quantize từng layer
quant_weights = {}
quant_scales = {}

for layer in model_f32.layers:
    if isinstance(layer, Conv2D) or isinstance(layer, Dense):
        w = layer.kernel.numpy()
        w_int8, scale = quantize_weights_symmetric(w, Q_MAX)
        quant_weights[layer.name] = w_int8
        quant_scales[layer.name] = scale
        print(f'{layer.name:<15} w_max={np.max(np.abs(w)):>8.4f} → scale={scale:>8.6f}')

print('\n✅ Weights quantized')

QUANTIZE WEIGHTS → INT8
Conv2D_1        w_max=  0.2570 → scale=0.002023
Conv2D_2        w_max=  0.4525 → scale=0.003563
Dense_1         w_max=  0.5909 → scale=0.004653
Dense_2         w_max=  0.5412 → scale=0.004262

✅ Weights quantized


## 🔍 8. Find Optimal Shift Bits (CRITICAL STEP)


In [18]:
print('='*70)
print('FIND OPTIMAL SHIFT BITS & QUANTIZE BIAS')
print('='*70)

# Extract weights and biases
conv1_w = quant_weights['Conv2D_1']
conv2_w = quant_weights['Conv2D_2']
dense1_w = quant_weights['Dense_1']
dense2_w = quant_weights['Dense_2']

conv1_b = model_f32.get_layer('Conv2D_1').bias.numpy()
conv2_b = model_f32.get_layer('Conv2D_2').bias.numpy()
dense1_b = model_f32.get_layer('Dense_1').bias.numpy()
dense2_b = model_f32.get_layer('Dense_2').bias.numpy()

conv1_scale = quant_scales['Conv2D_1']
conv2_scale = quant_scales['Conv2D_2']
dense1_scale = quant_scales['Dense_1']
dense2_scale = quant_scales['Dense_2']

print(f'Conv1 weight scale:  {conv1_scale:.6f}')
print(f'Conv2 weight scale:  {conv2_scale:.6f}')
print(f'Dense1 weight scale: {dense1_scale:.6f}')
print(f'Dense2 weight scale: {dense2_scale:.6f}')

# Calculate shift_bits dựa trên calibration range
shift_params = {}
for i, name in enumerate(layer_names_relu):
    act_range = calib_stats[name]['p99']
    act_scale = act_range / Q_MAX
    if name == 'Conv2D_1':
        w_scale = conv1_scale
    elif name == 'Conv2D_2':
        w_scale = conv2_scale
    else:
        w_scale = dense1_scale

    shift_float = np.log2(127 / (act_scale * w_scale + 1e-6))
    shift_bits = max(0, int(np.round(shift_float)))
    shift_params[name] = shift_bits
    print(f'{name:<12} act_range={act_range:>6.2f} → shift_bits={shift_bits}')

shift_params['Dense_2'] = None
print(f'Dense_2 output: KEEP INT32')

print('\n' + '-'*70)
print('🔥 SỬA LỖI BIAS QUANTIZATION (Scale Bias theo Scale MAC)')
print('-'*70)

# Tính Scale_Input của từng layer
scale_in_c1 = 1.0 / 127.0
scale_in_c2 = calib_stats['Conv2D_1']['p99'] / 127.0
scale_in_d1 = calib_stats['Conv2D_2']['p99'] / 127.0
scale_in_d2 = calib_stats['Dense_1']['p99'] / 127.0

# Tính Bias INT32 = Bias Float / (Scale_Input * Scale_Weight)
conv1_b_int32 = np.round(conv1_b / (scale_in_c1 * conv1_scale)).astype(np.int32)
conv2_b_int32 = np.round(conv2_b / (scale_in_c2 * conv2_scale)).astype(np.int32)
dense1_b_int32 = np.round(dense1_b / (scale_in_d1 * dense1_scale)).astype(np.int32)
dense2_b_int32 = np.round(dense2_b / (scale_in_d2 * dense2_scale)).astype(np.int32)

print(f"Ví dụ Conv1 Bias (Float vs INT32): {conv1_b[0]:.4f} -> {conv1_b_int32[0]}")
print("✅ Bias đã được lượng tử hóa chính xác!")

FIND OPTIMAL SHIFT BITS & QUANTIZE BIAS
Conv1 weight scale:  0.002023
Conv2 weight scale:  0.003563
Dense1 weight scale: 0.004653
Dense2 weight scale: 0.004262
Conv2D_1     act_range=  0.17 → shift_bits=25
Conv2D_2     act_range=  0.45 → shift_bits=23
Dense_1      act_range=  6.15 → shift_bits=19
Dense_2 output: KEEP INT32

----------------------------------------------------------------------
🔥 SỬA LỖI BIAS QUANTIZATION (Scale Bias theo Scale MAC)
----------------------------------------------------------------------
Ví dụ Conv1 Bias (Float vs INT32): 0.0279 -> 1748
✅ Bias đã được lượng tử hóa chính xác!


## 🧪 9. INT8 Inference Simulation


In [19]:
def quantize_input_int8(x_float32):
    return np.round(x_float32 * 127).astype(np.int8)

def relu_int8(x):
    return np.maximum(x, 0)

def conv2d_int8_simple(x_int8, w_int8, b_int32, shift_bits=7):
    H, W, Cin = x_int8.shape
    kH, kW = w_int8.shape[0], w_int8.shape[1]
    Cout = w_int8.shape[3]

    Ho = H - kH + 1
    Wo = W - kW + 1
    out = np.zeros((Ho, Wo, Cout), dtype=np.int32)
    w_int32 = w_int8.astype(np.int32)

    # Dùng np.einsum để tăng tốc độ nhân ma trận, chống treo máy
    for h in range(Ho):
        for w in range(Wo):
            patch = x_int8[h:h+kH, w:w+kW, :].astype(np.int32)
            out[h, w, :] = np.einsum('hwc,hwco->o', patch, w_int32) + b_int32

    out_shifted = out >> shift_bits
    out_int8 = np.clip(out_shifted, 0, 127).astype(np.int8)
    return out_int8

def maxpool2d_int8(x_int8, pool_size=2):
    H, W, C = x_int8.shape
    Ho = H // pool_size
    Wo = W // pool_size
    out = np.zeros((Ho, Wo, C), dtype=np.int8)
    for h in range(Ho):
        for w in range(Wo):
            patch = x_int8[h*pool_size:(h+1)*pool_size, w*pool_size:(w+1)*pool_size, :]
            out[h, w, :] = np.max(patch, axis=(0, 1))
    return out

def dense_int8_simple(x_int8_flat, w_int8, b_int32, shift_bits=7):
    x = x_int8_flat.astype(np.int32)
    w = w_int8.astype(np.int32)

    acc = np.dot(x, w) + b_int32
    if shift_bits is not None:
        acc = acc >> shift_bits
        acc = np.clip(acc, 0, 127).astype(np.int8)
    return acc

print('✅ INT8 simulation functions ready (Optimized)')

✅ INT8 simulation functions ready (Optimized)


## 🎯 10. Test Different Shift Bits (Grid Search)


In [20]:
print('='*70)
print('GRID SEARCH: Find Best Shift Bits')
print('='*70)

shift_ranges = {
    'Conv2D_1': range(4, 12),
    'Conv2D_2': range(4, 12),
    'Dense_1': range(4, 12)
}

best_acc = 0
best_shifts = {}
results = []

# Đã tối ưu tốc độ ở Cell 9, nên có thể tăng sample lên 150 để tìm Shift chính xác hơn
test_sample_count = min(150, len(X_test))
X_test_small = X_test[:test_sample_count]
y_test_small = y_test[:test_sample_count]

print(f'\n[Searching on {test_sample_count} samples... this may take a few minutes]')

for shift_c1 in shift_ranges['Conv2D_1']:
    for shift_c2 in shift_ranges['Conv2D_2']:
        for shift_d1 in shift_ranges['Dense_1']:
            correct = 0
            for idx in range(test_sample_count):
                x = X_test_small[idx]
                y_true = np.argmax(y_test_small[idx])

                x_q = quantize_input_int8(x)

                # Gọi hàm với biến b_int32 đã được tính ở Cell 8
                c1 = conv2d_int8_simple(x_q, conv1_w, conv1_b_int32, shift_c1)
                c1 = relu_int8(c1)
                c1 = maxpool2d_int8(c1, 2)

                c2 = conv2d_int8_simple(c1, conv2_w, conv2_b_int32, shift_c2)
                c2 = relu_int8(c2)

                c2_flat = c2.flatten()

                d1 = dense_int8_simple(c2_flat, dense1_w, dense1_b_int32, shift_d1)
                d1 = relu_int8(d1)

                d2 = dense_int8_simple(d1, dense2_w, dense2_b_int32, shift_bits=None)

                pred = np.argmax(d2)
                if pred == y_true:
                    correct += 1

            acc = correct / test_sample_count
            results.append({
                'shift_c1': shift_c1, 'shift_c2': shift_c2, 'shift_d1': shift_d1, 'accuracy': acc
            })

            if acc > best_acc:
                best_acc = acc
                best_shifts = {'Conv2D_1': shift_c1, 'Conv2D_2': shift_c2, 'Dense_1': shift_d1}

results.sort(key=lambda x: x['accuracy'], reverse=True)

print(f'\n🎯 Top 10 Shift Bit Combinations (on {test_sample_count} test samples):')
print(f'{"Conv1":>7} {"Conv2":>7} {"Dense1":>8} {"Accuracy":>10}')
print('-'*40)
for r in results[:10]:
    print(f'{r["shift_c1"]:>7} {r["shift_c2"]:>7} {r["shift_d1"]:>8} {r["accuracy"]*100:>9.2f}%')

print(f'\n✅ Best shifts: Conv1={best_shifts["Conv2D_1"]}, Conv2={best_shifts["Conv2D_2"]}, Dense1={best_shifts["Dense_1"]}')

GRID SEARCH: Find Best Shift Bits

[Searching on 150 samples... this may take a few minutes]

🎯 Top 10 Shift Bit Combinations (on 150 test samples):
  Conv1   Conv2   Dense1   Accuracy
----------------------------------------
      4      10       11    100.00%
      4      11       10    100.00%
      4      11       11    100.00%
      5       9       11    100.00%
      5      10       10    100.00%
      5      10       11    100.00%
      5      11        9    100.00%
      5      11       10    100.00%
      5      11       11    100.00%
      6       9       11    100.00%

✅ Best shifts: Conv1=4, Conv2=10, Dense1=11


## ✅ 11. Verify on Full Test Set


In [21]:
print('='*70)
print('FINAL VERIFICATION on Full Test Set')
print('='*70)

shift_c1 = best_shifts['Conv2D_1']
shift_c2 = best_shifts['Conv2D_2']
shift_d1 = best_shifts['Dense_1']

def simulate_int8_inference(x_float, s_c1, s_c2, s_d1):
    x_q = quantize_input_int8(x_float)
    # Truyền bias INT32 đã sửa
    c1 = conv2d_int8_simple(x_q, conv1_w, conv1_b_int32, s_c1)
    c1 = relu_int8(c1)
    c1 = maxpool2d_int8(c1, 2)

    c2 = conv2d_int8_simple(c1, conv2_w, conv2_b_int32, s_c2)
    c2 = relu_int8(c2)
    c2_flat = c2.flatten()

    d1 = dense_int8_simple(c2_flat, dense1_w, dense1_b_int32, s_d1)
    d1 = relu_int8(d1)

    d2 = dense_int8_simple(d1, dense2_w, dense2_b_int32, shift_bits=None)
    return d2

print(f'\nRunning INT8 inference on {len(X_test)} test samples...')

f32_preds = []
int8_preds = []
agreement_count = 0
int8_correct = 0

for idx in range(len(X_test)):
    x = X_test[idx]
    y_true = np.argmax(y_test[idx])

    f32_logits = model_f32.predict(np.expand_dims(x, 0), verbose=0)[0]
    f32_pred = np.argmax(f32_logits)

    int8_logits = simulate_int8_inference(x, shift_c1, shift_c2, shift_d1)
    int8_pred = np.argmax(int8_logits)

    f32_preds.append(f32_pred)
    int8_preds.append(int8_pred)

    if f32_pred == int8_pred:
        agreement_count += 1
    if int8_pred == y_true:
        int8_correct += 1

f32_correct = np.sum(np.array(f32_preds) == np.argmax(y_test, axis=1))
f32_acc = f32_correct / len(X_test)
int8_acc = int8_correct / len(X_test)
agreement_rate = agreement_count / len(X_test)
acc_drop = (f32_acc - int8_acc) * 100

print(f'\n' + '='*50)
print(f'Float32 Accuracy: {f32_acc*100:.2f}%')
print(f'INT8    Accuracy: {int8_acc*100:.2f}%')
print(f'Agreement Rate:   {agreement_rate*100:.2f}%')
print(f'Accuracy Drop:    {acc_drop:.2f}%')
print('='*50)

if acc_drop < 2:
    print(f'\n✅ EXCELLENT! Accuracy drop < 2%')
elif acc_drop < 5:
    print(f'\n⚠️  Good, but can improve. Try different shifts or calibration.')
else:
    print(f'\n❌ POOR. Shift bits still need tuning. Review calibration method.')

FINAL VERIFICATION on Full Test Set

Running INT8 inference on 883 test samples...

Float32 Accuracy: 99.21%
INT8    Accuracy: 98.87%
Agreement Rate:   98.75%
Accuracy Drop:    0.34%

✅ EXCELLENT! Accuracy drop < 2%


## 💾 12. Export to Hardware Format


In [22]:
print('='*70)
print('EXPORT WEIGHTS INT8 → Hardware (HEX, JSON, Excel)')
print('='*70)

hw_dir = os.path.join(OUTPUT_DIR, 'hardware_fixed')
os.makedirs(hw_dir, exist_ok=True)

# Convert biases to INT8
def bias_to_int32(b_float32):
    """Quantize bias to INT32 for hardware"""
    return np.round(b_float32).astype(np.int32)

conv1_b_int32 = bias_to_int32(conv1_b)
conv2_b_int32 = bias_to_int32(conv2_b)
dense1_b_int32 = bias_to_int32(dense1_b)
dense2_b_int32 = bias_to_int32(dense2_b)

# Export as JSON
output_json = {
    'model_name': 'CNN_Malware_PTQ_INT8_FIXED',
    'shift_bits': {
        'Conv2D_1': int(shift_c1),
        'Conv2D_2': int(shift_c2),
        'Dense_1': int(shift_d1),
        'Dense_2': None
    },
    'quantization_scales': {
        'Conv2D_1': float(conv1_scale),
        'Conv2D_2': float(conv2_scale),
        'Dense_1': float(dense1_scale),
        'Dense_2': float(dense2_scale)
    },
    'performance': {
        'float32_accuracy': float(f32_acc),
        'int8_accuracy': float(int8_acc),
        'accuracy_drop_percent': float(acc_drop)
    },
    'layer_shapes': {
        'Conv2D_1_weights': list(conv1_w.shape),
        'Conv2D_2_weights': list(conv2_w.shape),
        'Dense_1_weights': list(dense1_w.shape),
        'Dense_2_weights': list(dense2_w.shape)
    }
}

json_path = os.path.join(hw_dir, 'model_params.json')
with open(json_path, 'w') as f:
    json.dump(output_json, f, indent=2)

print(f'\n✅ Exported: {json_path}')
print(json.dumps(output_json, indent=2))

EXPORT WEIGHTS INT8 → Hardware (HEX, JSON, Excel)

✅ Exported: /content/drive/MyDrive/CNX/hardware_fixed/model_params.json
{
  "model_name": "CNN_Malware_PTQ_INT8_FIXED",
  "shift_bits": {
    "Conv2D_1": 4,
    "Conv2D_2": 10,
    "Dense_1": 11,
    "Dense_2": null
  },
  "quantization_scales": {
    "Conv2D_1": 0.002023482695221901,
    "Conv2D_2": 0.0035630108322948217,
    "Dense_1": 0.004652618896216154,
    "Dense_2": 0.00426159892231226
  },
  "performance": {
    "float32_accuracy": 0.9920724801812004,
    "int8_accuracy": 0.9886749716874292,
    "accuracy_drop_percent": 0.3397508493771184
  },
  "layer_shapes": {
    "Conv2D_1_weights": [
      3,
      3,
      1,
      16
    ],
    "Conv2D_2_weights": [
      3,
      3,
      16,
      32
    ],
    "Dense_1_weights": [
      416,
      48
    ],
    "Dense_2_weights": [
      48,
      2
    ]
  }
}


## 📊 13. Summary + Recommendations


In [24]:
print('\n' + '='*70)
print('SUMMARY - PTQ INT8 QUANTIZATION FIXED')
print('='*70)

print(f'''\n📌 RESULTS:
  Float32 Model Accuracy:  {f32_acc*100:.2f}%
  INT8    Model Accuracy:  {int8_acc*100:.2f}%
  Accuracy Drop:           {acc_drop:.2f}%

🔧 OPTIMAL SHIFT BITS (from grid search):
  Conv2D_1:  {shift_c1}
  Conv2D_2:  {shift_c2}
  Dense_1:   {shift_d1}
  Dense_2:   None (keep INT32 for output)

📂 EXPORTED FILES:
  ✅ {json_path}
  ✅ Conv1 weights: {conv1_w.shape} INT8
  ✅ Conv2 weights: {conv2_w.shape} INT8
  ✅ Dense1 weights: {dense1_w.shape} INT8
  ✅ Dense2 weights: {dense2_w.shape} INT8
''')

print('\n🎯 NEXT STEPS FOR HARDWARE:')
print('''  1. Load weights from JSON
  2. Set SHIFT_BITS in hardware
  3. Implement INT8 MACs:
     acc_int32 = sum(input_int8 * weight_int8) + bias_int32
  4. Shift right: acc_int32 >> SHIFT_BITS
  5. ReLU: max(0, shifted_value)
  6. Output Dense2 stays INT32 (no shift)
  7. Compare logits: if logits[0] > logits[1] → BENIGN else MALWARE
''')

if acc_drop < 1:
    print('\n✅ EXCELLENT RESULT! Ready for deployment.')
elif acc_drop < 5:
    print('\n⚠️  GOOD result. Monitor accuracy in production.')
else:
    print('\n❌ Need further optimization:')
    print('   - Try KL-divergence calibration')
    print('   - Use INT16 intermediate activations')
    print('   - Increase training epochs')
    print('   - Fine-tune with QAT')


SUMMARY - PTQ INT8 QUANTIZATION FIXED

📌 RESULTS:
  Float32 Model Accuracy:  99.21%
  INT8    Model Accuracy:  98.87%
  Accuracy Drop:           0.34%
  
🔧 OPTIMAL SHIFT BITS (from grid search):
  Conv2D_1:  4
  Conv2D_2:  10
  Dense_1:   11
  Dense_2:   None (keep INT32 for output)

📂 EXPORTED FILES:
  ✅ /content/drive/MyDrive/CNX/hardware_fixed/model_params.json
  ✅ Conv1 weights: (3, 3, 1, 16) INT8
  ✅ Conv2 weights: (3, 3, 16, 32) INT8
  ✅ Dense1 weights: (416, 48) INT8
  ✅ Dense2 weights: (48, 2) INT8


🎯 NEXT STEPS FOR HARDWARE:
  1. Load weights from JSON
  2. Set SHIFT_BITS in hardware
  3. Implement INT8 MACs:
     acc_int32 = sum(input_int8 * weight_int8) + bias_int32
  4. Shift right: acc_int32 >> SHIFT_BITS
  5. ReLU: max(0, shifted_value)
  6. Output Dense2 stays INT32 (no shift)
  7. Compare logits: if logits[0] > logits[1] → BENIGN else MALWARE


✅ EXCELLENT RESULT! Ready for deployment.


In [25]:
print('='*70)
print('CELL 12 — EXPORT MODEL CONFIG TO JSON')
print('='*70)

# Khởi tạo thông tin cấu hình tổng hợp
output_json = {
    'model_name': 'CNN_Malware_PTQ_INT8_FIXED_CHI_YOU',
    'shift_bits': {
        'Conv2D_1': int(shift_c1),
        'Conv2D_2': int(shift_c2),
        'Dense_1': int(shift_d1),
        'Dense_2': None
    },
    'quantization_scales': {
        'Conv2D_1': float(conv1_scale),
        'Conv2D_2': float(conv2_scale),
        'Dense_1': float(dense1_scale),
        'Dense_2': float(dense2_scale)
    },
    'performance': {
        'float32_accuracy': float(f32_acc),
        'int8_accuracy': float(int8_acc),
        'accuracy_drop_percent': float(acc_drop)
    },
    'layer_shapes': {
        'Conv2D_1_weights': list(conv1_w.shape),
        'Conv2D_2_weights': list(conv2_w.shape),
        'Dense_1_weights': list(dense1_w.shape),
        'Dense_2_weights': list(dense2_w.shape)
    }
}

# Tạo cấu trúc lưu trữ chi tiết bao gồm cả ma trận trọng số
weights_dict = {
    'method': 'PTQ (Chi-You style - Fixed)',
    'quantization': 'INT8 symmetric per-tensor',
    'shift_params': {k: int(v) if v is not None else None for k, v in shift_params.items()},
    'Conv2D_1': {'weights': conv1_w.tolist(), 'bias': conv1_b_int32.tolist(), 'shift': int(shift_c1)},
    'Conv2D_2': {'weights': conv2_w.tolist(), 'bias': conv2_b_int32.tolist(), 'shift': int(shift_c2)},
    'Dense_1': {'weights': dense1_w.tolist(), 'bias': dense1_b_int32.tolist(), 'shift': int(shift_d1)},
    'Dense_2': {'weights': dense2_w.tolist(), 'bias': dense2_b_int32.tolist(), 'shift': None}
}

print("✅ Đã chuẩn bị xong dữ liệu JSON cấu hình!")

CELL 12 — EXPORT MODEL CONFIG TO JSON
✅ Đã chuẩn bị xong dữ liệu JSON cấu hình!


In [26]:
# Cấu hình thư mục đầu ra phần cứng cố định
hw_dir = os.path.join(OUTPUT_DIR, 'hardware_fixed')
os.makedirs(hw_dir, exist_ok=True)

# Ghi file thông số cấu hình JSON
json_param_path = os.path.join(hw_dir, 'model_params.json')
with open(json_param_path, 'w') as f:
    json.dump(output_json, f, indent=2)

# Ghi file trọng số đầy đủ JSON
json_weights_path = os.path.join(hw_dir, 'weights_int8.json')
with open(json_weights_path, 'w') as f:
    json.dump(weights_dict, f, indent=2)

print(f'✅ Đã khởi tạo thư mục đầu ra phần cứng thành công!')
print(f'💾 Cấu hình lưu tại: {json_param_path}')
print(f'💾 Trọng số lưu tại: {json_weights_path}')

✅ Đã khởi tạo thư mục đầu ra phần cứng thành công!
💾 Cấu hình lưu tại: /content/drive/MyDrive/CNX/hardware_fixed/model_params.json
💾 Trọng số lưu tại: /content/drive/MyDrive/CNX/hardware_fixed/weights_int8.json


In [27]:
print('='*70)
print('CELL 13.1 — PACK WEIGHTS & BIAS TO HEX FOR BRAM')
print('='*70)

# Khớp định dạng shape phần cứng (kH, kW, Cin, Cout) -> (Cout, Cin, kH, kW) cho các lớp Conv
conv1_w_hw = conv1_w.transpose(3, 2, 0, 1)
conv2_w_hw = conv2_w.transpose(3, 2, 0, 1)

def export_hex(weights_int8, bias_int32, filename):
    """
    Hàm đóng gói ghép nối Weights INT8 và Bias INT32 về mảng byte số nguyên,
    sau đó dồn bọc 4 byte thành một từ mã 32-bit (word) dạng mã HEX.
    """
    # Đồng bộ ép Bias về dải quy ước để nạp khối nhớ BRAM theo phong cách Chi-You
    bias_packed = np.clip(bias_int32, -127, 127).astype(np.int8)

    # Nối phẳng toàn bộ mảng dữ liệu
    flat = np.concatenate([weights_int8.flatten(), bias_packed.flatten()]).astype(np.int8)

    # Bù thêm 0 (padding) để đảm bảo chia hết cho 4 byte (32-bit word)
    pad = (4 - len(flat) % 4) % 4
    flat = np.concatenate([flat, np.zeros(pad, dtype=np.int8)])

    lines = []
    for i in range(0, len(flat), 4):
        b0, b1, b2, b3 = flat[i:i+4].view(np.uint8)
        # Tổ hợp thành từ mã 32-bit (Little Endian tương thích phần cứng)
        word = (int(b3) << 24) | (int(b2) << 16) | (int(b1) << 8) | int(b0)
        lines.append(f'{word:08x}')

    path = os.path.join(hw_dir, filename)
    with open(path, 'w') as f:
        f.write('\n'.join(lines))
    return path, len(lines)

# Tiến hành xuất các file HEX lưu trữ cho các khối BRAM tương ứng từng lớp
p1, n1 = export_hex(conv1_w_hw, conv1_b_int32, 'bram_w1.hex')
p2, n2 = export_hex(conv2_w_hw, conv2_b_int32, 'bram_w2.hex')
p3, n3 = export_hex(dense1_w,   dense1_b_int32, 'bram_w3.hex')
p4, n4 = export_hex(dense2_w,   dense2_b_int32, 'bram_w4.hex')

print('Danh sách file HEX cấu trúc bộ nhớ BRAM đã xuất:')
for p, n, desc in [(p1, n1, 'Conv2D_1'), (p2, n2, 'Conv2D_2'), (p3, n3, 'Dense_1'), (p4, n4, 'Dense_2')]:
    size_kb = os.path.getsize(p) / 1024
    print(f'  ✅ {os.path.basename(p):<18s} [{size_kb:>6.1f} KB]  {n:<6} dòng lệnh (32-bit words) — {desc}')

CELL 13.1 — PACK WEIGHTS & BIAS TO HEX FOR BRAM
Danh sách file HEX cấu trúc bộ nhớ BRAM đã xuất:
  ✅ bram_w1.hex        [   0.4 KB]  40     dòng lệnh (32-bit words) — Conv2D_1
  ✅ bram_w2.hex        [  10.2 KB]  1160   dòng lệnh (32-bit words) — Conv2D_2
  ✅ bram_w3.hex        [  44.0 KB]  5004   dòng lệnh (32-bit words) — Dense_1
  ✅ bram_w4.hex        [   0.2 KB]  25     dòng lệnh (32-bit words) — Dense_2


In [29]:
import pandas as pd

print('='*70)
print('CELL 13.2 — EXPORT WEIGHTS TO EXCEL FOR DEBUGGING')
print('='*70)

excel_path = os.path.join(hw_dir, 'weights_int8_ptq.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Bảng lưu thông số dịch bit (Shift bits)
    df_shift = pd.DataFrame([
        {'Layer': 'Conv2D_1', 'Shift_Bits': shift_c1},
        {'Layer': 'Conv2D_2', 'Shift_Bits': shift_c2},
        {'Layer': 'Dense_1', 'Shift_Bits': shift_d1},
        {'Layer': 'Dense_2', 'Shift_Bits': 'None (Keep INT32)'}
    ])
    df_shift.to_excel(writer, sheet_name='Shift_Params', index=False)

    # Biến đổi phẳng các lớp Conv để lưu dạng ma trận 2D trong Excel
    pd.DataFrame(conv1_w_hw.reshape(conv1_w_hw.shape[0], -1)).to_excel(writer, sheet_name='Conv1_W', index=True)
    pd.DataFrame(conv2_w_hw.reshape(conv2_w_hw.shape[0], -1)).to_excel(writer, sheet_name='Conv2_W', index=True)
    pd.DataFrame(dense1_w).to_excel(writer, sheet_name='Dense1_W', index=True)
    pd.DataFrame(dense2_w).to_excel(writer, sheet_name='Dense2_W', index=True)

print(f'✅ File bảng tính Excel gỡ lỗi đã được xuất thành công tại:\n   ➔ {excel_path}')

CELL 13.2 — EXPORT WEIGHTS TO EXCEL FOR DEBUGGING
✅ File bảng tính Excel gỡ lỗi đã được xuất thành công tại:
   ➔ /content/drive/MyDrive/CNX/hardware_fixed/weights_int8_ptq.xlsx


In [104]:
print('='*70)
print('CELL 14 — TEST 5 RANDOM SAMPLES & EXPORT INPUT HEX FOR HARDWARE')
print('='*70)

test_dir = os.path.join(OUTPUT_DIR, 'test_fixed')
os.makedirs(test_dir, exist_ok=True)
label_names = ['Benign', 'Malware']

# Lấy toàn bộ chỉ mục của từng nhãn trong tập Test
all_benign_idx = np.where(y_test[:, 0] == 1)[0]
all_malware_idx = np.where(y_test[:, 1] == 1)[0]

# Bốc ngẫu nhiên 2 mẫu Benign và 3 mẫu Malware
random_benign = np.random.choice(all_benign_idx, 2, replace=False)
random_malware = np.random.choice(all_malware_idx, 3, replace=False)

# Gộp lại và xáo trộn vị trí ngẫu nhiên
test_indices = np.concatenate([random_benign, random_malware])
np.random.shuffle(test_indices)

print('Kết quả chạy mô phỏng kiểm chứng trên 5 mẫu ảnh ngẫu nhiên:')
print('-'*65)

for case_no, idx in enumerate(test_indices):
    x_sample = X_test[idx]
    y_true_label = np.argmax(y_test[idx])

    # Gọi hàm mô phỏng inference INT8 chính xác (với Bias đã phân tỷ lệ)
    logits_sim = simulate_int8_inference(x_sample, shift_c1, shift_c2, shift_d1)
    pred_label = np.argmax(logits_sim)
    status_str = '✅ CHÍNH XÁC' if pred_label == y_true_label else '❌ SAI LỆCH'

    print(f'[Mẫu {case_no+1}] Chỉ mục={idx:<4} | Gốc={label_names[y_true_label]:<7} '
          f'| Dự đoán={label_names[pred_label]:<7} | Logits=[{logits_sim[0]:>6}, {logits_sim[1]:>6}] | {status_str}')

    # Ép dải điểm ảnh float [0, 1] về định dạng pixel số nguyên uint8 [0, 255] nguyên bản
    img_uint8 = np.round(x_sample.squeeze() * 255).astype(np.uint8)  # Kích thước khung (8, 32)
    flat_pixels = img_uint8.flatten()  # Chuỗi phẳng gồm 256 phần tử byte

    # Gom cụm đóng gói 4 pixel liên tiếp vào một từ mã 32-bit HEX
    input_hex_lines = []
    for i in range(0, len(flat_pixels), 4):
        p_block = flat_pixels[i:i+4]
        word_val = (int(p_block[3]) << 24) | (int(p_block[2]) << 16) | (int(p_block[1]) << 8) | int(p_block[0])
        input_hex_lines.append(f'{word_val:08x}')

    hex_out_path = os.path.join(test_dir, f'test{case_no+1}_in32.hex')
    with open(hex_out_path, 'w') as f:
        f.write('\n'.join(input_hex_lines))

print('-'*65)
print(f'✅ Đã kết xuất xong 5 file HEX dữ liệu ảnh đầu vào mẫu tại thư mục:\n   ➔ {test_dir}')

CELL 14 — TEST 5 RANDOM SAMPLES & EXPORT INPUT HEX FOR HARDWARE
Kết quả chạy mô phỏng kiểm chứng trên 5 mẫu ảnh ngẫu nhiên:
-----------------------------------------------------------------
[Mẫu 1] Chỉ mục=587  | Gốc=Malware | Dự đoán=Malware | Logits=[-65430,  70652] | ✅ CHÍNH XÁC
[Mẫu 2] Chỉ mục=639  | Gốc=Malware | Dự đoán=Malware | Logits=[-19996,  32294] | ✅ CHÍNH XÁC
[Mẫu 3] Chỉ mục=355  | Gốc=Benign  | Dự đoán=Benign  | Logits=[ 43041, -20514] | ✅ CHÍNH XÁC
[Mẫu 4] Chỉ mục=455  | Gốc=Benign  | Dự đoán=Benign  | Logits=[ 43295, -20758] | ✅ CHÍNH XÁC
[Mẫu 5] Chỉ mục=687  | Gốc=Malware | Dự đoán=Benign  | Logits=[ 42778, -20358] | ❌ SAI LỆCH
-----------------------------------------------------------------
✅ Đã kết xuất xong 5 file HEX dữ liệu ảnh đầu vào mẫu tại thư mục:
   ➔ /content/drive/MyDrive/CNX/test_fixed


In [105]:
print('\n' + '=' * 70)
print('CELL 15 — SUMMARY & HARDWARE INFERENCE SPECIFICATION')
print('=' * 70)

# Khảo sát sự tồn tại thực tế của các file trong thư mục đầu ra
print("Kiểm tra trạng thái sẵn sàng của các cấu kiện phần cứng:")
print('-'*65)
for base_file in ['model_params.json', 'weights_int8.json', 'weights_int8_ptq.xlsx',
                  'bram_w1.hex', 'bram_w2.hex', 'bram_w3.hex', 'bram_w4.hex']:
    full_target_path = os.path.join(hw_dir, base_file)
    indicator = '✅ SẴN SÀNG' if os.path.exists(full_target_path) else '❌ THIẾU'
    file_size_str = f'{os.path.getsize(full_target_path)/1024:.1f} KB' if os.path.exists(full_target_path) else '0 KB'
    print(f'  {indicator}  ➔ {base_file:<22s} Dung lượng: [{file_size_str:>8s}]')

print('-'*65)
print(f'📌 THÔNG SỐ SHIFT BITS ĐÃ TỐI ƯU HÓA:')
print(f'  • Lớp Conv2D_1 : Dịch chuyển phải >> {shift_c1} bits')
print(f'  • Lớp Conv2D_2 : Dịch chuyển phải >> {shift_c2} bits')
print(f'  • Lớp Dense_1  : Dịch chuyển phải >> {shift_d1} bits')
print(f'  • Lớp Dense_2  : Giữ nguyên dải dữ liệu INT32 (Không dịch dịch bit)')

print('\n🎯 CÁC BƯỚC THIẾT KẾ KHỐI MAC TRÊN PHẦN CỨNG (VERILOG / VHDL):')
instruction_text = f'''  1. Lượng tử hóa đầu vào: Điểm ảnh uint8 đọc từ file mẫu sẽ được dịch chuyển
     phải 1 bit (pixel_int8 = pixel_uint8 >> 1) để đưa về dải [0, 127] đồng bộ.
  2. Lớp Conv1: Thực hiện phép nhân tích lũy (MAC) số nguyên giữa Input INT8 và
     Trọng số INT8, cộng thêm Bias INT32 trực tiếp trong bộ tích lũy.
     Thực hiện dịch phải dữ liệu >> {shift_c1} bits, áp dụng hàm max(0, giá_trị)
     cho ReLU và ép kiểu bão hòa về INT8 đầu ra.
  3. Lớp MaxPool: Lấy giá trị lớn nhất trong cửa sổ 2x2 mà không làm thay đổi dải số.
  4. Lớp Conv2: Tiến hành tính toán MAC, cộng Bias INT32 -> Dịch phải >> {shift_c2} bits
     -> Lọc ReLU -> Cắt bão hòa trả về INT8.
  5. Lớp Dense1: Thực hiện nhân ma trận, cộng Bias INT32 -> Dịch phải >> {shift_d1} bits
     -> Lọc ReLU -> Trả về mảng dữ liệu INT8.
  6. Lớp Dense2 (Ngõ ra): Thực hiện nhân ma trận và cộng Bias INT32. Giữ nguyên giá trị
     trong thanh ghi INT32, tuyệt đối không dịch chuyển bit để tránh làm mất độ phân giải phân lớp.
  7. Khối quyết định: So sánh giá trị đại số giữa logit[0] (Benign) và logit[1] (Malware).
     Nếu logit[0] > logit[1] kết luận là ảnh An toàn, ngược lại là Mã độc.'''
print(instruction_text)


CELL 15 — SUMMARY & HARDWARE INFERENCE SPECIFICATION
Kiểm tra trạng thái sẵn sàng của các cấu kiện phần cứng:
-----------------------------------------------------------------
  ✅ SẴN SÀNG  ➔ model_params.json      Dung lượng: [  0.7 KB]
  ✅ SẴN SÀNG  ➔ weights_int8.json      Dung lượng: [325.3 KB]
  ✅ SẴN SÀNG  ➔ weights_int8_ptq.xlsx  Dung lượng: [106.1 KB]
  ✅ SẴN SÀNG  ➔ bram_w1.hex            Dung lượng: [  0.4 KB]
  ✅ SẴN SÀNG  ➔ bram_w2.hex            Dung lượng: [ 10.2 KB]
  ✅ SẴN SÀNG  ➔ bram_w3.hex            Dung lượng: [ 44.0 KB]
  ✅ SẴN SÀNG  ➔ bram_w4.hex            Dung lượng: [  0.2 KB]
-----------------------------------------------------------------
📌 THÔNG SỐ SHIFT BITS ĐÃ TỐI ƯU HÓA:
  • Lớp Conv2D_1 : Dịch chuyển phải >> 4 bits
  • Lớp Conv2D_2 : Dịch chuyển phải >> 10 bits
  • Lớp Dense_1  : Dịch chuyển phải >> 11 bits
  • Lớp Dense_2  : Giữ nguyên dải dữ liệu INT32 (Không dịch dịch bit)

🎯 CÁC BƯỚC THIẾT KẾ KHỐI MAC TRÊN PHẦN CỨNG (VERILOG / VHDL):
  1. Lượng tử